# Market Language Model: baseline training

Pipeline: load candles -> candle-geometry features with trailing volatility normalization -> triple-barrier labels with timeout and conservative tie-break -> non-overlapping sampling with strict temporal splits -> LightGBM baseline -> cost-adjusted expectancy.

Inputs deliberately exclude indicators, absolute prices, and coin identity. The cross-asset test at the end checks whether any edge generalizes or is asset-specific. Replace the synthetic generator with real OHLCV before drawing conclusions; synthetic random-walk data has no learnable edge by construction and is only here to prove the pipeline runs end to end.

In [ ]:
import mlm
from mlm import (generate_synthetic_ohlcv, FeatureConfig, BarrierConfig,
                 SplitConfig, build_dataset, temporal_split,
                 train_lightgbm, predict_win_prob, evaluate_strategy)
from mlm.metrics import CostConfig

feat_cfg = FeatureConfig(vol_window=100)
bar_cfg = BarrierConfig(tp=0.02, sl=0.01, horizon=48)
split_cfg = SplitConfig(window=64, non_overlap=True)
cost_cfg = CostConfig(fee=0.0005, slippage=0.0005)

In [ ]:
# Phase 1 asset (stand-in for BTC). Swap for load_ohlcv_csv('btc_5m.csv').
df = generate_synthetic_ohlcv(n=60000, seed=0, vol=0.002)
X, y, idx = build_dataset(df, feat_cfg, bar_cfg, split_cfg)
splits = temporal_split(X, y, idx)
print('samples', len(X))
for k, (Xs, ys, _) in splits.items():
    import numpy as np
    print(k, len(Xs), 'win_rate', round(float((ys == mlm.WIN).mean()), 4))

In [ ]:
model = train_lightgbm(splits['train'], splits['val'])

In [ ]:
# In-asset test performance (held-out future of the same asset).
Xte, yte, _ = splits['test']
prob = predict_win_prob(model, Xte)
res = evaluate_strategy(yte, prob, bar_cfg.tp, bar_cfg.sl, cost_cfg=cost_cfg)
res

In [ ]:
# Cross-asset generalization: a different volatility scale, no retraining.
df_other = generate_synthetic_ohlcv(n=60000, seed=1, vol=0.006)
Xo, yo, _ = build_dataset(df_other, feat_cfg, bar_cfg, split_cfg)
prob_o = predict_win_prob(model, Xo)
evaluate_strategy(yo, prob_o, bar_cfg.tp, bar_cfg.sl, cost_cfg=cost_cfg)

Read `expectancy` as mean return per taken trade after round-trip costs. Positive and stable across both the in-asset test and the cross-asset test is the goal; on synthetic random-walk data expect roughly zero, which confirms the pipeline is honest rather than leaking. The breakeven win-rate in the result is the no-skill hurdle to beat.